In [182]:
import gzip
import itertools
from collections import defaultdict 
import numpy as np
import os

## Load in the palindrome library, and make a mapping of target to sequence

In [87]:
def base_to_index(x):
    if x == 'A':
        return 0;
    elif x == 'C':
        return 1;
    elif x == 'G':
        return 2;
    elif x == 'T':
        return 3;
    return 4;

def target_pileup(guide,target):
    assert len(guide) == len(target)
    alt_base = np.zeros((5,len(guide)))
    differences = 0
    for (index,(gbase,tbase)) in enumerate(zip(guide,target)):
        alt_base[base_to_index(tbase),index] += 1 
        if gbase != tbase:
            differences += 1
    prop_diff = float(differences)/float(len(guide))
    return(alt_base)

target_pileup("GTATAGGTGATCACCTATAC","GTCTAGGTGATCACCTATAC")

array([[0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0.,
        1., 0., 1., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 1., 0.,
        0., 0., 0., 1.],
       [1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 1.,
        0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.]])

In [156]:
stats = "../data/data/pipeline_output/MF_ABPal1/MF_ABPal1.stats"
#stats = open("../data/data/pipeline_output/MF_Pal1_Ct/MF_Pal1_Ct.stats")

# pal1 = "GGTGACCGTTAACGGTCACCAGGA"
# pal3 = "GGTGACCGGTACCGGTCACCAGGA"
pal = "GGTGACCGTTAACGGTCACCAGGA"


def process_palindrome(stats_file):
    all_counts = np.zeros((5,len(pal1)))
    stats = open(stats_file)
    
    for line in stats:
        sp = line.strip("\n").split("\t")
        if sp[1] == "PASS" and not '-' in sp[23] and len(pal) == len(sp[23]):
            all_counts = np.add(all_counts,target_pileup(pal,sp[23]))

    return(all_counts)

def counts_to_pct(palindrome, all_counts, i):
    return(all_counts[base_to_index(pal[i]),i]/sum(all_counts[:,i]))

def output_file(all_counts,sample, output, palindrome_sequence, is_umi, is_control, pal_tag, control_wt):
    for i in range(len(pal)):
        output.write(sample + "\t" + str(i) + "\t" + 
                     pal[i] + "\t" + "\t".join([str(x) for x in all_counts[:,i]]) + 
                     "\t" + str(counts_to_pct(palindrome_sequence,all_counts, i)) + 
                     "\t" + str(is_umi) + "\t" + str(is_control) + "\t" + pal_tag + "\t" +
                     str(counts_to_pct(palindrome_sequence,control_wt, i)) + "\n")

# We expect editing in the follow places:
# ABE 7.10 A->G
#  CE      C->T
# A->G .      G .          G
# C->T .     T AA .       T TT
# pal1 = "GGTGACCGTTAACGGTCACCAGGA"
# pal3 = "GGTGACCGGTACCGGTCACCAGGA"
pal = "GGTGACCGTTAACGGTCACCAGGA"
def process_palindrome(stats_file):
    all_counts = np.zeros((5,len(pal1)))
    stats = open(stats_file)
    
    for line in stats:
        sp = line.strip("\n").split("\t")
        if sp[1] == "PASS" and not '-' in sp[23] and len(pal) == len(sp[23]):
        

    return(all_counts)


In [158]:
inputs = ["../data//data/pipeline_output/MF_UMIBEPal1/MF_UMIBEPal1.stats",
"../data//data/pipeline_output/MF_BEPal1/MF_BEPal1.stats",
"../data//data/pipeline_output/MF_Pal3_Ct/MF_Pal3_Ct.stats",
"../data//data/pipeline_output/MF_UMIPal3/MF_UMIPal3.stats",
"../data//data/pipeline_output/MF_UMIABPal3/MF_UMIABPal3.stats",
"../data//data/pipeline_output/MF_ABPal3/MF_ABPal3.stats",
"../data//data/pipeline_output/MF_BEPal3/MF_BEPal3.stats",
"../data//data/pipeline_output/MF_UMIBEPal3/MF_UMIBEPal3.stats",
"../data//data/pipeline_output/MF_UMIPal1/MF_UMIPal1.stats",
"../data//data/pipeline_output/MF_ABPal1/MF_ABPal1.stats",
"../data//data/pipeline_output/MF_Pal1_Ct/MF_Pal1_Ct.stats",
"../data//data/pipeline_output/MF_UMIABPal1/MF_UMIABPal1.stats"]

# parse out which palindrome each target is
pals = [pal1 if "Pal1" in x else pal3 for x in inputs]

output = open("../data/all_samples.txt","w")
output.write("sample\tindex\tbase\tA\tC\tG\tT\tN\twt\tumi\tcontrol\tpal_tag\tcontrol_wt\n")
    
pal1_control = process_palindrome("../data//data/pipeline_output/MF_Pal1_Ct/MF_Pal1_Ct.stats")
pal3_control = process_palindrome("../data//data/pipeline_output/MF_Pal3_Ct/MF_Pal3_Ct.stats")

for pal, input_fl in zip(pals, inputs):
    sample = input_fl.split("/")[-1].split(".")[0]
    print(sample)
    is_umi = "UMI" in sample 
    is_control = "Ct" in sample
    pal_tag = "pal1" if "Pal1" in sample else "pal3"
    
    counts = process_palindrome(input_fl)
    if pal_tag == "pal1":
        output_file(counts,sample,output,pal1,is_umi,is_control,pal_tag,pal1_control)
    else:
        output_file(counts,sample,output,pal3,is_umi,is_control,pal_tag,pal3_control)
        
output.close()
output_norm.close()

MF_UMIBEPal1
MF_BEPal1
MF_Pal3_Ct
MF_UMIPal3
MF_UMIABPal3
MF_ABPal3
MF_BEPal3
MF_UMIBEPal3
MF_UMIPal1
MF_ABPal1
MF_Pal1_Ct
MF_UMIABPal1


In [136]:
# we like the looks of the editing in a few samples. Lets see if the editing in unidirectional in 
# a few of samples (editing in only one site or another)

In [207]:
### each set position is a location where a base should flip from a AT to a CG or reverse
def orient_palindrome_GC_to_AT(left_positions, right_positions, guide, sequence):
    index = 0
    edited = []
    for gbase,sbase in zip(guide,sequence):
        if gbase != sbase and (gbase == "C" or gbase == "G") and (sbase == "A" or sbase == "T"):
            edited.append(index)
        index += 1
    
    # now find the orientation -- are we left, right, both, or neither?
    left_in = sum([1 if x in left_positions else 0 for x in edited]) > 0
    right_in = sum([1 if x in right_positions else 0 for x in edited]) > 0
    
    direction = "NONE"
    if left_in and right_in:
        direction = "BOTH"
    elif left_in:
        direction = "LEFT"
    elif right_in:
        direction = "RIGHT"
    return((left_in,right_in,direction))

### each set position is a location where a base should flip from a AT to a CG or reverse
def orient_palindrome_AT_to_GC(left_positions, right_positions, guide, sequence):
    index = 0
    edited = []
    for gbase,sbase in zip(guide,sequence):
        if gbase != sbase and (gbase == "A" or gbase == "T") and (sbase == "C" or sbase == "G"):
            edited.append(index)
        index += 1
    
    # now find the orientation -- are we left, right, both, or neither?
    left_in = sum([1 if x in left_positions else 0 for x in edited]) > 0
    right_in = sum([1 if x in right_positions else 0 for x in edited]) > 0
    
    direction = "NONE"
    if left_in and right_in:
        direction = "BOTH"
    elif left_in:
        direction = "LEFT"
    elif right_in:
        direction = "RIGHT"
    return((left_in,right_in,direction))
       

def process_palindrome(stats_file, lefts, rights, guide, output_file, at_to_gc):
    stats = open(stats_file)
    left_counts = 0
    right_counts = 0
    total_count = 0
    raw_count = 0
    counts = {"NONE": 0, "BOTH": 0, "LEFT": 0, "RIGHT": 0}
    
    for line in stats:
        raw_count += 1
        sp = line.strip("\n").split("\t")
        if sp[1] == "PASS" and not '-' in sp[23] and len(pal) == len(sp[23]):
            total_count += 1
            if at_to_gc:
                orient = orient_palindrome_AT_to_GC(lefts,rights,guide,sp[23])
            else:
                orient = orient_palindrome_GC_to_AT(lefts,rights,guide,sp[23])
            counts[orient[2]] = counts.get(orient[2],0) + 1
            left_counts += orient[0]
            right_counts += orient[1]
    
    
    output_file.write(os.path.basename(stats_file).strip("stats").replace(".", "") + "\t" + str(at_to_gc) + "\t" + guide + "\t" + 
                      str(min(lefts)) + "\t" + str(len(lefts)) + "\t" + 
                      str(min(rights)) + "\t" + str(len(rights)) + "\t" + 
                      str(total_count) + "\t" + str(left_counts) + "\t" + str(right_counts) + "\t" +
                      str(counts["NONE"]) + "\t" + str(counts["BOTH"]) + "\t" + 
                      str(counts["LEFT"]) + "\t" + str(counts["RIGHT"]) + "\t" + str(raw_count) + "\n")

output_file = open("palindrome_table.txt","w")
output_file.write("file\tat_to_gc\tguide\tleft_min\tleft_len\tright_min\tright_len\ttotal_reads\tleft_muts\tright_muts\tnone\tboth\tleft\tright\traw_reads\n")

pal1 = "GGTGACCGTTAACGGTCACCAGGA"
pal3 = "GGTGACCGGTACCGGTCACCAGGA"


for pal, input_fl in zip(pals, inputs):
    is_umi = "UMI" in input_fl 
    is_control = "Ct" in input_fl
    pal_tag = "pal1" if "Pal1" in input_fl else "pal3"
    at_to_gc = not (True if "BE" in input_fl else False)
    print(input_fl + "\t" + str(at_to_gc))
    lengths = range(2,8)
    for length in lengths:
        lefts = range(0,length) 
        rights = range(20 - length,20)
        if is_control:
            process_palindrome(input_fl, lefts, rights, pal1, output_file,True)
            process_palindrome(input_fl, lefts, rights, pal1, output_file,False)
        else:
            if pal_tag == "pal1":
                process_palindrome(input_fl, lefts, rights, pal1, output_file,at_to_gc)
            else:
                process_palindrome(input_fl, lefts, rights, pal3, output_file,at_to_gc)

output_file.close()
counts

../data//data/pipeline_output/MF_UMIBEPal1/MF_UMIBEPal1.stats	False
../data//data/pipeline_output/MF_BEPal1/MF_BEPal1.stats	False
../data//data/pipeline_output/MF_Pal3_Ct/MF_Pal3_Ct.stats	True
../data//data/pipeline_output/MF_UMIPal3/MF_UMIPal3.stats	True
../data//data/pipeline_output/MF_UMIABPal3/MF_UMIABPal3.stats	True
../data//data/pipeline_output/MF_ABPal3/MF_ABPal3.stats	True
../data//data/pipeline_output/MF_BEPal3/MF_BEPal3.stats	False
../data//data/pipeline_output/MF_UMIBEPal3/MF_UMIBEPal3.stats	False
../data//data/pipeline_output/MF_UMIPal1/MF_UMIPal1.stats	True
../data//data/pipeline_output/MF_ABPal1/MF_ABPal1.stats	True
../data//data/pipeline_output/MF_Pal1_Ct/MF_Pal1_Ct.stats	True
../data//data/pipeline_output/MF_UMIABPal1/MF_UMIABPal1.stats	True


{'NONE': 134105, 'LEFT': 1741, 'BOTH': 547, 'RIGHT': 3584}